# Determinism under parallelism

For a fixed package version, the same table and configuration produce
byte-identical reports across runs and thread counts when neither timing
nor a deadline is enabled. Fixed row selection, stable ordering, and
deterministic aggregation enforce this contract.

## The rules

1. **Wall-clock effects are opt-in.** A `deadline_seconds` limit can
   change which passes run; it never interrupts one mid-computation.
   `timing=True` records elapsed times in coverage without changing
   findings. Both options can make serialized reports vary between runs.
2. **Seeds everywhere randomness is used.** The ξ tie-breaking shuffle,
   the GBM holdout shuffles, the stratified sampler's stratum offsets —
   all are fixed arithmetic or fixed-seed generators derived from
   table shape and column indices, never from entropy sources.
3. **Sorted, keyed collections in the report path.** The report tree is
   built from `BTreeMap`s and sorted vectors; the JSON writer walks them
   in key order. Key-order iteration is insertion-order-independent.
4. **Stable tie-breaking everywhere.** Every sort in the pipeline that
   can see ties has a final tie-break on a stable key (a column name, an
   index) so "equal" candidates keep a fixed order.
5. **Deterministic reduction of parallel results.** Rayon work happens
   per column, per pair, per target — results are collected into
   position-indexed containers or merged through ordered maps, never
   accumulated into a shared float in arrival order. Where floating-point
   sums matter (concentration), the summation order is the sorted key
   order, not the parallel traversal order.

## Why parallelism does not break it

Parallel passes compute one result per item and collect results in input
order. Aggregation follows collection rather than occurring in a shared
accumulator whose value depends on scheduling. Where a
global ranking happens (dependence findings, rule selection), it sorts
by (score, stable keys) before truncation, so tied items are ordered
consistently when applying a cap.

The stratified sampler `sample_rows(n, cap)`
picks row `start + mix(stratum) mod stratum_width` per stratum. A plain
stride (`row = k·j`) would also be deterministic — and could repeatedly
sample one weekday out of a weekly cycle. Hashed offsets vary the
position within each stratum while remaining deterministic ([](../methodology/sampling.ipynb)).

## What opt-in nondeterminism looks like

A deadline makes the completed work depend on runtime. It is checked
between passes: a running pass completes, then later passes are skipped
if the deadline has elapsed. Coverage records `"deadline reached"` for
the skipped passes. Without a deadline and with timing disabled, thread
count and equivalent input chunk layouts do not change the report bytes.

In [ ]:
import polars as pl

import onceover

df = pl.DataFrame({
    "a": [1, 2, 2, 3, None, 3],
    "b": [1.5, 2.5, None, 2.5, 7.0, 1.5],
})
first = onceover.profile(df)
second = onceover.profile(df, budget="standard")
print("byte-identical reruns:", first.to_json() == second.to_json())